# Lab 09: BERT Fine-tuning

**Module 09** | Read `notes/09-bert.pdf` before running this notebook.

Fine-tune DistilBERT on SST-2 sentiment with the Hugging Face Trainer API and report validation accuracy.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


## DistilBERT fine-tuning on SST-2

BERT-style encoders produce contextual embeddings for every token. For classification we attach a small head on the `[CLS]` representation and fine-tune the whole stack on labeled sentences. **DistilBERT** retains most of the accuracy of BERT-base with roughly half the parameters, which makes it a practical default for instructional fine-tuning.

SST-2 is a binary sentiment benchmark (positive vs negative movie reviews). We load `SetFit/sst2` and use the Hugging Face `Trainer` API so training, evaluation, and metric logging stay declarative.


### Step 1: Load raw SST-2 examples

Inspect a few raw sentences and labels before any tokenization. Label `0` means negative sentiment, label `1` means positive.


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
BATCH_SIZE = 8
LABEL_NAMES = {0: "negative", 1: "positive"}

raw = load_dataset("SetFit/sst2")
print("Dataset splits:", list(raw.keys()))
print(f"  Train:      {len(raw['train']):,} examples")
print(f"  Validation: {len(raw['validation']):,} examples")

print("\nFirst 3 raw training examples")
for i in range(3):
    row = raw["train"][i]
    print(f"  [{i}] label={row['label']} ({LABEL_NAMES[row['label']]})")
    print(f"      text: {row['text'][:120]}{'...' if len(row['text']) > 120 else ''}")


### Step 2: Tokenize and decode one example

The tokenizer maps text to `input_ids` and an `attention_mask`. Sequences are padded and truncated to 128 tokens. Decoding non-padding tokens confirms subword splitting (e.g., "loved" may become "love" + "##d").


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

example_text = raw["train"][0]["text"]
encoded_one = tokenizer(
    example_text,
    truncation=True,
    max_length=MAX_LENGTH,
    padding="max_length",
)

input_ids = encoded_one["input_ids"]
attention_mask = encoded_one["attention_mask"]
active_ids = [tid for tid, mask in zip(input_ids, attention_mask) if mask == 1]
tokens = tokenizer.convert_ids_to_tokens(active_ids)

print("Tokenization demo (training example 0)")
print(f"  Text: {example_text}")
print(f"  Label: {raw['train'][0]['label']} ({LABEL_NAMES[raw['train'][0]['label']]})")
print(f"  Active token count: {len(active_ids)}")
print(f"  Tokens: {tokens}")
print(f"  Decoded: {tokenizer.decode(active_ids)}")

def tokenize_batch(batch: dict) -> dict:
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

encoded = raw.map(tokenize_batch, batched=True)
encoded = encoded.rename_column("label", "labels")
encoded.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print(f"\nEncoded columns: {encoded['train'].column_names}")


### Step 3: Load model and count trainable parameters

We attach a two-class classification head on top of DistilBERT. All weights are fine-tuned (nothing frozen) for this short one-epoch demo.


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model summary")
print(f"  Model:             {MODEL_NAME}")
print(f"  Device:            {device}")
print(f"  Total parameters:  {total_params:,}")
print(f"  Trainable params:  {trainable_params:,}")
print(f"  Frozen params:     {total_params - trainable_params:,}")


### Step 4: Train for one epoch and evaluate

`TrainingArguments` sets one epoch, batch size 8, and evaluation after training. `Trainer` handles the optimization loop, device placement, and metric aggregation.


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    accuracy = (preds == labels).mean()
    return {"accuracy": float(accuracy)}


training_args = TrainingArguments(
    output_dir=str(ROOT / "labs" / "outputs" / "distilbert_sst2"),
    num_train_epochs=1,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
metrics = trainer.evaluate()

print("Training results")
print(f"  Training loss:        {train_result.training_loss:.4f}")
print(f"  Validation accuracy:  {metrics['eval_accuracy']:.4f} ({metrics['eval_accuracy']:.1%})")
print(f"  Validation loss:      {metrics['eval_loss']:.4f}")


### Step 5: Predict five validation sentences

We run the fine-tuned model on five held-out reviews and tabulate text, true label, predicted label, and softmax confidence. Compare mistakes with ambiguous phrasing.


In [ ]:
model.eval()
n_preds = 5
val_rows = raw["validation"].select(range(n_preds))

pred_table = []
for i in range(n_preds):
    text = val_rows[i]["text"]
    true_label = val_rows[i]["label"]
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    ).to(device)

    with torch.no_grad():
        logits = model(**inputs).logits
        probs = F.softmax(logits, dim=-1)[0]
        pred_label = int(probs.argmax().item())
        confidence = float(probs[pred_label].item())

    pred_table.append({
        "text": text[:80] + ("..." if len(text) > 80 else ""),
        "true_label": LABEL_NAMES[true_label],
        "pred_label": LABEL_NAMES[pred_label],
        "confidence": f"{confidence:.1%}",
        "correct": pred_label == true_label,
    })

pred_df = pd.DataFrame(pred_table)
print("Validation predictions (first 5 examples)")
print(pred_df.to_string(index=False))
print(f"\nAccuracy on these 5: {pred_df['correct'].mean():.0%}")
